# Advanced Tools

Dataset generation, filtering, tokenizer fertility, checkpoint selection, and inference internals. Run these after you have completed the [training notebook](02_train_from_scratch.ipynb).

## Setup

In [ ]:
import sys, os
from pathlib import Path

# Ensure uv is available in cloud notebook runtimes such as RunPod.
!pip install -q uv

if "google.colab" in sys.modules or "RUNPOD_POD_ID" in os.environ:
    if not Path("Makefile").exists():
        !git clone --depth 1 https://github.com/khordoo/dusty-lm.git
        os.chdir("dusty-lm")
    !pip install -q -e .
    !make download-datasets
elif not Path("Makefile").exists():
    # Walk up so make commands find the Makefile from any CWD
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "Makefile").exists():
            os.chdir(str(parent))
            break
    else:
        print("Run this from the repo root:\n        uv sync")


## Overview

This notebook covers:

1. **Generate SFT Data** - create synthetic conversations using an external LLM, then preview, filter, and flatten
2. **Tokenizer Fertility** - measure tokens per word across vocabulary sizes
3. **Select Pretraining Checkpoints** - evaluate, compare, and promote the best base model
4. **Select SFT Checkpoints** - evaluate, compare, and promote the best fine-tuned model
5. **Inference Internals** - chat completion API, context windows, and convenience wrappers


## Inspect the Dataset Files

The training loop expects raw data under `artifacts/datasets/` and tokenized datasets beside it.

In [ ]:
from pathlib import Path

for path in [
    Path("artifacts/datasets/tinystories_base.txt"),
    Path("artifacts/datasets/dusty_sft.jsonl"),
]:
    if path.exists():
        print(f"\u2713 {path}: {path.stat().st_size / 1024 / 1024:.2f} MB")
    else:
        print(f"\u2717 {path}")

## 1. Generate SFT Data


This step uses an external LLM to generate synthetic Dusty conversations via an OpenAI-compatible API.

**Default provider:** OpenRouter (`https://openrouter.ai/api/v1`). Gives access to many models under a single API key.

You can point `OPENAI_BASE_URL` at any OpenAI-compatible endpoint (OpenAI, Together AI, vLLM, Ollama, LM Studio, etc.). Just make sure the model is capable enough for the task.

**Models:** The primary model and fallback are set via `DUSTY_MODEL` and `DUSTY_FALLBACK_MODEL`. Override them in the `make` command below.

**Before you start:**
- Set `OPENAI_API_KEY` to your API key (OpenRouter, OpenAI, or any OpenAI-compatible provider)
- Set `OPENAI_BASE_URL` to override the default OpenRouter endpoint (e.g. `https://api.openai.com/v1`)
- If you change the persona, update the prompt and categories in `data_pipeline/generate_sft.py`

**Cost:** The full dataset (500 examples per category, ~30k rows) cost us under $3 with Qwen. Start with 5 per category (as below) to check quality before scaling up.

In [ ]:
# Optional: set your API key and provider before running this cell.
# import os
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"  # default

!make synthesize-sft DUSTY_SFT_PER_CATEGORY=5 DUSTY_SFT_BATCH_SIZE=5

### 1.1 Preview SFT Rows


Each JSONL row needs `category`, `user`, and `dusty`. The `category` field is metadata for filtering and balancing; training uses the user and assistant text.

In [ ]:
import json
from pathlib import Path

sft_path = Path("artifacts/datasets/dusty_sft.jsonl")
with sft_path.open("r", encoding="utf-8") as file:
    for index, line in zip(range(5), file):
        row = json.loads(line)
        print(json.dumps(row, indent=2)[:800])
        print()

### 1.2 Filter and Balance SFT Data [Optional]


The model uses `max_seq_len=256` tokens. If a training conversation exceeds the limit, the tail gets truncated and the model never sees the `<eos>` (End of Sequence) token. Without learning when to emit `<eos>`, the model will generate endless, looping text during inference.

The main way to keep examples short is to write the generation prompt so the LLM produces concise responses (2-3 sentences). Filtering is an optional safeguard: it drops any example whose answer exceeds `DUSTY_SFT_MAX_ANSWER_TOKENS` and samples a category-balanced subset. Use a smaller target for experiments and a larger target for final training.

In [ ]:
!make filter-sft \
  DUSTY_SFT_FILTER_TARGET=200 \
  DUSTY_SFT_MAX_ANSWER_TOKENS=256 \
  DUSTY_SFT_FILTERED_OUT=artifacts/datasets/dusty_sft_200.jsonl

## 2. Tokenizer Fertility


Fertility is tokens per whitespace-delimited word. For a tiny model, roughly `1.2` to `1.5` is usually a useful range. Higher values mean words are split heavily; lower values may spend too much of the model on embeddings.

Vocabulary size is part of the model's parameter budget. A larger vocabulary increases the embedding and output projection matrices, which can make the checkpoint larger without making the transformer layers more capable. Fertility testing asks a practical question: if we spend more parameters on vocabulary, do we actually reduce token count enough to justify it?

| Vocab Size | Lines | Words | Tokens | Fertility |
|-----------|------:|------:|------:|---------:|
| 4,096 | 10,000 | 327,204 | 430,793 | 1.317 |
| 8,192 | 10,000 | 327,204 | 410,990 | 1.256 |

The table confirms the diminishing returns:

- **4,096 tokens:** 1.32 fertility at a small parameter footprint
- **8,192 tokens:** 1.26 fertility, but adds roughly 4M embedding parameters

For Dusty, moving from 4K to 8K did not produce a large enough improvement to justify the extra parameters, so the default tokenizer stays around 4K tokens.

In [ ]:
!uv run python data_pipeline/test_tokenizer_fertility.py \
  --input artifacts/datasets/tinystories_base.txt \
  --training-files artifacts/datasets/tinystories_base.txt \
  --lines 10000 \
  --vocab-sizes 2048 4096 8192

## 3. Select Pretraining Checkpoints

### 3.1 Evaluate Pretraining Checkpoints

Before SFT, evaluate the base `dusty8m` checkpoints directly. The pretrained model was trained on TinyStories, so it should produce coherent story-like text but does not yet have the Dusty conversational personality.

In [ ]:
!make generate PROFILE=dusty8m PROMPT="Deep in the forest there was a " TOP_P=0.9 TEMPERATURE=0.8

### 3.2 Compare Pretraining Checkpoints


During pretraining, the loss curve does not tell the whole story. A model with a low loss might generate looping, repetitive text, while an earlier checkpoint produces perfectly readable English. Production pipelines run full benchmark suites to quantify this, but here we keep it simple with direct side-by-side reads.

To find the best version of your base model, evaluate the generated text manually. Use `CHECKPOINT_STEP` to sample outputs from different intermediate checkpoints. Generate a short story from several steps and compare them for coherence, spelling, and repetition.

In [ ]:
# Use story-based cliffhangers to test the raw pre-trained base model
base_prompts = [
    "Once upon a time,",
    "Lily was a little girl who loved to",
    "Deep in the forest, there lived a little boy named Timmy. One day, he",
]
base_checkpoint_steps = [200, 250, 300]

for step in base_checkpoint_steps:
    print("#" * 80)
    print("BASE CHECKPOINT_STEP:", step)
    for prompt in base_prompts:
        print("=" * 80)
        print("PROMPT:", prompt)
        !make generate PROFILE=dusty8m CHECKPOINT_STEP={step} PROMPT="{prompt}" TOP_P=0.9 TEMPERATURE=0.8

Sampling these checkpoints serves two different purposes depending on where you are in the training cycle:

- **Early Debugging (The Sanity Check)**

    While the model is actively training, sample early steps to verify that it is progressively learning. Early checkpoints will show the model transition from pure noise to forming real letters to generating basic words. Keep Chinchilla scaling laws in mind - do not expect cohesive stories this early. You are simply confirming that the training pipeline is working.

- **Finding the Golden Zone (Post-Training)**

    Once training is complete, compare the checkpoints that were actually created by your run. With the current 100k TinyStories, batch 224, 1-epoch setup, this means checking the later checkpoints around steps 200 to 300.

**Note on our run:** To keep this demonstration lightweight, the current golden path uses a 100k TinyStories slice, batch size 224, and 1 pretraining epoch. In that setup, step 300 is the target baseline for a functional base model.

Sample several checkpoints strictly within your chosen region and compare them. The best checkpoint is the one that produces the most natural-sounding text with the fewest spelling errors and the least amount of looping.


### 3.3 Promote the Best Pretraining Checkpoint


Once you have evaluated your checkpoints and selected the winner from your pretraining run, you must **promote** it to be the official base model.

The subsequent Supervised Fine-Tuning (SFT) phase is hardcoded to look for a specific base model file named `artifacts/checkpoints/dusty8m.pt`. To promote your best checkpoint, copy your chosen step file and rename it to this target filename.

Run the following command to promote your chosen step (replace `300` if you selected a different one):

In [ ]:
!cp artifacts/checkpoints/dusty8m_step_300.pt artifacts/checkpoints/dusty8m.pt

**Sanity Check Your Base Model**

After promoting the file, it is highly recommended to run one final sanity check. This ensures that you copied the correct checkpoint and that `dusty8m.pt` is successfully generating coherent English before you invest time into the fine-tuning phase.

Run the generation command using the base profile to test your newly promoted model:

In [ ]:
!make generate PROFILE=dusty8m PROMPT="Deep in the forest there was a" TOP_P=0.9 TEMPERATURE=0.8

## 4. Select SFT Checkpoints

### 4.1 Compare SFT Step Checkpoints


Loss is only a coarse signal for tiny character models. During training, step checkpoints such as `artifacts/checkpoints/dusty8m_sft_step_100.pt` let you evaluate intermediate model behavior. Generate the same fixed prompt set from several `CHECKPOINT_STEP` values and compare instruction following, persona consistency, repetition, and answer length.

Unlike the pre-training phase, Supervised Fine-Tuning (SFT) does not follow Chinchilla scaling laws. Instead of aiming for hundreds of millions of tokens, SFT is simply about teaching the base model a specific conversational format and persona. For this repository, we use a custom ~35k instruction dataset. With batch size 224, 2 epochs produces roughly 300 iterations and gives a strong lightweight demo.

During SFT, the model can overfit to the chat format quickly. This is why you should stop relying purely on the loss curve and begin selecting checkpoints qualitatively by reading the generated text.

When a step checkpoint behaves better than the final checkpoint, promote it by copying that file over the profile's default checkpoint path. After promotion, remove `CHECKPOINT_STEP` from the generation command; normal `make generate PROFILE=sft_dusty8m ...` will use the selected model.

## 4.1 Compare SFT Checkpoints

Just as we established during pre-training, the loss curve is only a coarse signal. However, what you are evaluating for during SFT is entirely different. Instead of checking for basic English coherence, you are testing for **instruction following, persona consistency, and appropriate answer length**.


Unlike pre-training, SFT does not follow Chinchilla scaling laws; you are simply teaching the model a specific conversational format. Because small models can quickly overfit to chat templates, qualitative testing is critical. For this repository, we use a custom ~35k instruction dataset. With the current batch 224, 2-epoch setup, step 250 is the checkpoint to compare against nearby steps.

Sample intermediate checkpoints strictly within this region using `CHECKPOINT_STEP` to find the best persona fit.

In [ ]:
prompts = [
    "where are you?",
    "are you scared?",
    "what do you dream about?",
]

checkpoint_steps = [100, 500, 1000]  # Update these based on your runs

for step in checkpoint_steps:
    print("#" * 80)
    print("CHECKPOINT_STEP:", step)
    for prompt in prompts:
        print("=" * 80)
        print("PROMPT:", prompt)
        !make generate PROFILE=sft_dusty8m CHECKPOINT_STEP={step} PROMPT="{prompt}" TOP_P=0.9 TEMPERATURE=0.8

### 4.2 Promote the Selected SFT Checkpoint

Exactly as you did with the base model, you must promote your winning checkpoint to finalize the pipeline. This copies your chosen step file and renames it to `artifacts/checkpoints/dusty8m_sft.pt`, which is the default weight for the `sft_dusty8m` profile used in the chat interface.

Set `BEST_STEP` to the checkpoint step that produced the best qualitative behavior. Run the following commands to promote your model and perform a final sanity check (replace `250` if you chose a different step):

In [ ]:
BEST_STEP = 250

!cp artifacts/checkpoints/dusty8m_sft_step_{BEST_STEP}.pt artifacts/checkpoints/dusty8m_sft.pt

# This command intentionally omits CHECKPOINT_STEP. It now uses the promoted default checkpoint.
!make generate PROFILE=sft_dusty8m PROMPT="where are you?" TOP_P=0.9 TEMPERATURE=0.8

## 5. Inference Internals

### 5.1 Load an Inference Engine

Use the local promoted SFT checkpoint if you trained one. If you skipped training, pre-trained checkpoints are available on Hugging Face Hub at `mkhordoo/dusty-8m-sft` and will be downloaded automatically when you run the cell below.

In [ ]:
from pathlib import Path
import torch
from dustylm.inference import Inference

checkpoint_path = Path("artifacts/checkpoints/dusty8m_sft.pt")
tokenizer_path = Path("artifacts/tokenizers/dusty_tokenizer.json")

if not checkpoint_path.exists():
    from huggingface_hub import hf_hub_download

    print("Local checkpoint not found, downloading from Hugging Face Hub...")
    checkpoint_path = Path(
        hf_hub_download(repo_id="mkhordoo/dusty-8m-sft", filename="dusty8m_sft.pt")
    )
    tokenizer_path = Path(
        hf_hub_download(
            repo_id="mkhordoo/dusty-8m-sft", filename="dusty_tokenizer.json"
        )
    )

engine = Inference(
    checkpoint_path=checkpoint_path,
    tokenizer_path=tokenizer_path,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("Profile:", engine.profile_name)

### 5.2 Convenience Wrapper and CLI Chat


For scripts or demos, wrap the response dictionary and return only the generated assistant text. For a terminal chat loop, use `make chat`, which runs `uv run python -m dustylm.inference`.

In [ ]:
def chat(prompt, *, max_tokens=128, temperature=0.8, top_p=0.8):
    response = engine.chat_completion(
        [{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in [
    "who are you?",
    "are you scared?",
    "what do you dream about?",
]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")

### 5.3 OpenAI-Style Chat Completion


`Inference.chat_completion()` accepts OpenAI-style `messages` and returns an OpenAI-like dictionary. That keeps local DustyLM inference close to common chat-completion calling patterns while still running against a local PyTorch checkpoint.

In [ ]:
messages = [{"role": "user", "content": "where are you?"}]

response = engine.chat_completion(
    messages=messages,
    max_tokens=64,
    temperature=0.8,
    top_p=0.8,
)

response

In [ ]:
assistant_text = response["choices"][0]["message"]["content"]
print("Dusty>", assistant_text)

### 5.4 Short Context Windows for Tiny Models


Dusty is only about 8M parameters, so the default chat history window is intentionally tiny: `max_chat_turns=1`. That keeps the current user request from being diluted by old conversation tokens. Larger SFT profiles, such as SmolLM2-based models, can use a larger window.

In [ ]:
print("Profile:", engine.profile_name)
print("Default max_chat_turns:", engine.spec.max_chat_turns)

history = [
    {"role": "system", "content": "Answer as Dusty, a tiny vacuum robot."},
    {"role": "user", "content": "what is under the couch?"},
    {"role": "assistant", "content": "crumbs. many crumbs."},
    {"role": "user", "content": "should you go there?"},
]

response = engine.chat_completion(history, max_tokens=128)
print("Default window:", response["choices"][0]["message"]["content"])

response = engine.chat_completion(history, max_tokens=128, max_chat_turns=2)
print("Two-turn window:", response["choices"][0]["message"]["content"])

### 5.5 The dustylm SDK

You just loaded the raw `.pt` weights, instantiated the `nn.Module`, and wrote the token-by-token generation loop from scratch. That is exactly how the engine works under the hood.

However, in a production environment, you don't want to rewrite that boilerplate every time you need to chat with your model. To solve this, we wrapped the exact logic you just learned into a lightweight Python package.

Now, you can run the model in just two lines. The SDK works with your own local checkpoints and with published Hub models:

```python
from dustylm import DustyLM

# Your own training run -- point to your local directory
model = DustyLM.from_pretrained("./dusty-final-checkpoint/")

# Or pull a published model from Hugging Face Hub
model = DustyLM.from_pretrained("mkhordoo/dusty-8m-sft")

response = model.chat([{"role": "user", "content": "who are you?"}])
print(response["choices"][0]["message"]["content"])
# beep. i am a little robot. i clean floors and find crumbs.
```

The SDK auto-detects the architecture from your checkpoint's state dict and supports both PyTorch and ONNX backends. Install it with `pip install dustylm`.

## 6. Model Evaluation

Instead of just chatting with Dusty, you can systematically test and score the model's performance across different training steps.

### 6a. Automated Checkpoint Comparison

The training notebook compares a few checkpoints manually. For larger runs, use the evaluation helper to run the same fixed inputs across multiple checkpoint steps and save the results for review.

The tracked input sets live in `evaluation/inputs/`:

- `base_inputs.json` checks base-model text completion behavior.
- `sft_inputs.json` checks Dusty persona, chat behavior, greetings, maker identity, and common edge cases.

The tool picks the input file automatically: profiles containing `sft` use the SFT inputs; other profiles use the base inputs. You can still override this with `EVAL_INPUT_SET`, `--input-set`, or a custom JSON file.

Generated reports are written to `artifacts/evaluations/checkpoints/` as both `.json` and `.csv` files. The JSON keeps run metadata; the CSV is easier to scan in a spreadsheet.

Run through `make`:

```bash
make eval-checkpoints EVAL_PROFILE=sft_dusty8m EVAL_STEPS="150 200 250" EVAL_TEMPERATURE=0.7 EVAL_TOP_P=0.9
```

Or run the Python script directly:

```bash
uv run python evaluation/compare_checkpoints.py --profile sft_dusty8m --steps 150 200 250 --temperature 0.7 --top-p 0.9
```

For base checkpoints, switch the profile and steps:

```bash
make eval-checkpoints EVAL_PROFILE=dusty8m EVAL_STEPS="200 250 300"
```

If you change Dusty's persona or train a different character, update the JSON inputs too. The model should be evaluated on questions that match the behavior you care about. Each input only needs an `id`, `category`, and `input` field, so the CSV output can be traced back to the exact question.

### 6b. Quantitative Scoring

If you want to move beyond visual comparison and quantitatively measure the model's performance, we have separated the scoring logic from the core training pipeline.

The `evaluation/` directory contains standalone CLI tools to systematically sweep checkpoints and score outputs for logical consistency, contradiction rates, and semantic depth.

**[Check out the Evaluation README](./evaluation/README.md) for the CLI commands and methodology.**
